# SFT Fine-Tuning: `google/translategemma-4b-it`
## Spanish → Catalan (Valencian Variety)

**Pipeline overview:**

1. Install dependencies
2. Load `google/translategemma-4b-it` with 4-bit quantization + LoRA
3. **SFT** (Supervised Fine-Tuning) on `gplsi/amic_parallel` (ES → VA)

Model available at HuggingFace: `guerreropaula/translategemma4b-sft-es-va2`



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 1. Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    sacrebleu \
    sentencepiece \
    huggingface_hub \
    flash-attn-triton

In [ ]:
%%capture
!pip install -q git+https://github.com/google-research/bleurt.git
!wget -q https://storage.googleapis.com/bleurt-oss/bleurt-base-128.zip
!unzip -q bleurt-base-128.zip

In [ ]:
import torch
import transformers, peft, trl

print(f"PyTorch      : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch      : 2.10.0+cu128
transformers : 5.0.0
peft         : 0.18.1
trl          : 0.29.0
CUDA         : True
GPU          : NVIDIA A100-SXM4-80GB
VRAM         : 85.1 GB


---
## 2. Hugging Face Login

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""
login(token=HF_TOKEN)

---
## 3. Load Model: `google/translategemma-4b-it` with 4-bit Quantization + LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_ID       = "google/translategemma-4b-it"
MAX_SEQ_LENGTH = 256
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16       = torch.cuda.is_bf16_supported()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16 if USE_BF16 else torch.float16,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Base model with 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = HF_TOKEN,
    dtype         = torch.bfloat16 if USE_BF16 else torch.float16,
    trust_remote_code   = True,
)

print(f"Model  : {MODEL_ID}")
print(f"Params : {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model  : google/translategemma-4b-it
Params : 2490M


In [ ]:
# Prepare for k-bit training and attach LoRA adapter
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


---
## 4. TranslateGemma Prompt Template

TranslateGemma uses a structured chat format where each user-turn message
includes the fields `type`, `source_lang_code`, `target_lang_code`, and `text`.
The model uses `"ca"` as the language code for Catalan/Valencian (no dialect differentiation).

In [ ]:
SOURCE_LANG_CODE = "es"
TARGET_LANG_CODE = "ca"   # TranslateGemma does not differentiate Valencian from Catalan
SOURCE_COL       = "ES"
TARGET_COL       = "VA"


def _make_messages(source_text: str) -> list:
    """Build the user-turn message list for the TranslateGemma template."""
    return [
        {
            "role": "user",
            "content": [
                {
                    "type"             : "text",
                    "source_lang_code" : SOURCE_LANG_CODE,
                    "target_lang_code" : TARGET_LANG_CODE,
                    "text"             : source_text,
                }
            ],
        }
    ]


def format_for_sft(source_text: str, target_text: str) -> str:
    """Full prompt + reference answer; used to build the SFT dataset."""
    prompt = tokenizer.apply_chat_template(
        _make_messages(source_text), tokenize=False, add_generation_prompt=True
    )
    return prompt + target_text + tokenizer.eos_token


def make_inference_prompt(source_text: str) -> str:
    """Prompt only (no answer); used for inference and GRPO."""
    return tokenizer.apply_chat_template(
        _make_messages(source_text), tokenize=False, add_generation_prompt=True
    )


# Sanity check
test_src = "El ayuntamiento ha aprobado el nuevo presupuesto."
test_tgt = "L'ajuntament ha aprovat el nou pressupost."
print("=== SFT format ===")
print(format_for_sft(test_src, test_tgt))
print("\n=== Inference / GRPO format ===")
print(make_inference_prompt(test_src))

=== SFT format ===
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities.
Produce only the Catalan translation, without any additional explanations or commentary. Please translate the following Spanish text into Catalan:


El ayuntamiento ha aprobado el nuevo presupuesto.<end_of_turn>
<start_of_turn>model
L'ajuntament ha aprovat el nou pressupost.<eos>

=== Inference / GRPO format ===
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities.
Produce only the Catalan translation, without any additional explanations or commentary. Please translate the following Spanish text into Catalan:


El ayuntamiento 

---
## 5. SFT Training Dataset: `gplsi/amic_parallel`

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("gplsi/amic_parallel")
print(raw_dataset)
print("\nExample:", raw_dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

data/train/va-es-documents.curated.jsonl:   0%|          | 0.00/404M [00:00<?, ?B/s]

data/train/va-es-paragraph.length.curate(…):   0%|          | 0.00/245M [00:00<?, ?B/s]

data/train/va-es-sentence.length.ner.cur(…):   0%|          | 0.00/91.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/680978 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['VA', 'ES', 'source'],
        num_rows: 680978
    })
})

Example: {'VA': "Música a la UPV\n\nDel pasdoble 'Agüero' de José Franco, al 'Bolero' de Ravel, gaudeix el divendres 11 del concert de la Banda Simfònica de la UPV a l'Alfons Roig (19.30 h)\n\nL'Auditori Alfons Roig de la Facultat de Belles Arts (edifici 3N, planta baixa) de la Universitat Politècnica de València (UPV) és l'escenari elegit per a la celebració, el pròxim divendres, 11 de març, d'un nou concert de la Banda Simfònica de la UPV.\nAmb accés lliure fins a completar l'aforament del recinte, l'agrupació musical universitària dirigida per Francisco J. Valero interpretarà, a partir de les 19.30 hores, un programa dividit en dues parts.\n\nObres de José Franco, Philip Wilby, Josep Alamà Gil, George Bizet i Maurice Ravel\n\nLa primera començarà a ritme de pasdoble, amb Agüero, de José Franco, peça a la qual seguiran el Concert per a bombardí de Philip Wilby -amb Jorge L

---
## 6. SFT Training

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForLanguageModeling

SFT_TRAIN_SAMPLES = 50_000
SFT_OUTPUT_DIR    = "./translategemma4b_sft_es_va"


def formatting_prompts_func(examples):
    return {"text": [
        format_for_sft(src, tgt)
        for src, tgt in zip(examples[SOURCE_COL], examples[TARGET_COL])
    ]}


sft_dataset = raw_dataset.map(
    formatting_prompts_func,
    batched        = True,
    remove_columns = raw_dataset["train"].column_names,
)
print("SFT dataset formatted. Example:")
print(sft_dataset["train"]["text"][0])

Map:   0%|          | 0/680978 [00:00<?, ? examples/s]

SFT dataset formatted. Example:
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities.
Produce only the Catalan translation, without any additional explanations or commentary. Please translate the following Spanish text into Catalan:


Música en la UPV

Del pasodoble 'Agüero' de José Franco al 'Bolero' de Ravel, disfruta el viernes 11 del concierto de la Banda Sinfónica UPV en el Alfons Roig (19.30 h)

El Auditori Alfons Roig de la Facultad de Bellas Artes (edificio 3N, planta baja) de la Universitat Politècnica de València (UPV) es el escenario elegido para la celebración, el próximo viernes 11 de marzo, de un nuevo concierto de la Banda Sinfónica UPV.
Con acceso libre hasta completar el aforo del recinto, la agrupación musical universitaria dirigida por Francisco J. Valero interpretará, a pa

In [ ]:
# Saves png tranining loss
from transformers import TrainerCallback
import matplotlib.pyplot as plt

class LossPlotCallback(TrainerCallback):
    def __init__(self, save_path="sft_loss_curve.png"):
        self.save_path = save_path
        self.steps  = []
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.steps.append(state.global_step)
            self.losses.append(logs["loss"])

            plt.figure(figsize=(10, 4))
            plt.plot(self.steps, self.losses, linewidth=1.5, color="#2B5797")
            plt.title("SFT Training Loss")
            plt.xlabel("Step")
            plt.ylabel("Loss")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(self.save_path, dpi=150, bbox_inches="tight")
            plt.close()

In [ ]:
class Gemma3DataCollator:
    """Wraps the standard causal-LM collator and injects token_type_ids
    (all zeros) required by Gemma 3's attention-mask function."""

    def __init__(self, tokenizer):
        self.base_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer, mlm=False
        )

    def __call__(self, features):
        batch = self.base_collator(features)
        batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        return batch


In [ ]:
model.train()

sft_trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = sft_dataset["train"].shuffle(seed=42).select(range(SFT_TRAIN_SAMPLES)),
    data_collator    = Gemma3DataCollator(tokenizer),
    callbacks        = [LossPlotCallback("sft_loss_curve.png")],
    args = SFTConfig(
        packing                     = False,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 32,
        warmup_steps                = 25,
        max_steps                   = 2000,
        learning_rate               = 2e-4,
        logging_steps               = 25,
        optim                       = "paged_adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = SFT_OUTPUT_DIR,
        save_steps                  = 25,
        report_to                   = "none",
        fp16                        = not USE_BF16,
        bf16                        = USE_BF16,
        gradient_checkpointing      = True,
        dataloader_num_workers      = 2,
    ),
)

gpu_start = round(torch.cuda.max_memory_reserved() / 1e9, 2)
print(f"VRAM before training: {gpu_start} GB")

VRAM before training: 15.12 GB


In [ ]:
torch.cuda.empty_cache()
import gc; gc.collect()

1003

In [ ]:
sft_stats = sft_trainer.train()

print(f"\nSFT training complete")
print(f"VRAM max: {round(torch.cuda.max_memory_reserved()/1e9,2)} GB")
print(f"Time     : {sft_stats.metrics['train_runtime']:.1f}s")
print(f"Final Loss : {sft_stats.metrics.get('train_loss', 'N/A'):.4f}")

VRAM antes de entrenar: 39.34 GB


Step,Training Loss
25,0.717626
50,0.688967
75,0.665579
100,0.650163
125,0.685803
150,0.673868
175,0.670064
200,0.680450
225,0.674921
250,0.686462



SFT completado
VRAM maxima: 39.39 GB
Tiempo     : 14968.1s
Loss final : 0.6899


---
## 7. Save and Push SFT Checkpoint

In [ ]:
SFT_ADAPTER_DIR = SFT_OUTPUT_DIR + "/lora_adapter"
model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"LoRA adapter saved to: {SFT_ADAPTER_DIR}")

In [ ]:
model.push_to_hub("guerreropaula/translategemma4b-sft-es-va2", token=HF_TOKEN)
tokenizer.push_to_hub("guerreropaula/translategemma4b-sft-es-va2", token=HF_TOKEN)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  61%|######    | 39.9MB / 65.7MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkemq_zl2/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/guerreropaula/translategemma4b-sft-es-va2/commit/6da04e04305f120a8df752757e19b6f320b0e69d', commit_message='Upload tokenizer', commit_description='', oid='6da04e04305f120a8df752757e19b6f320b0e69d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/guerreropaula/translategemma4b-sft-es-va2', endpoint='https://huggingface.co', repo_type='model', repo_id='guerreropaula/translategemma4b-sft-es-va2'), pr_revision=None, pr_num=None)